# ML Assignment 02

### Dataset: House Price Prediction Dataset

### Total Marks: 100

---

## Exam Instructions:

1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights


## Student Information


In [1]:
# Fill in your information
name = "Rafiur Rahman Saikat"  # Write your full name here
email = "md.rafiur.rahman.saikkat@gmail.com"  # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Rafiur Rahman Saikat
Email : md.rafiur.rahman.saikkat@gmail.com


---

## Question 1 (10 Marks)

Load the House Price dataset and display:

- Dataset shape
- First 10 rows
- 5 random samples


In [9]:
# Question 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("house_price_regression_dataset.csv")

df.shape
df.head(10)
df.sample(5)

display(df.shape, df.head(10), df.sample(5))

(1000, 8)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
199,4590,1,3,1958,2.768017,0,2,9.188937e+05
509,1323,4,1,1986,4.384110,1,6,3.627196e+05
778,2070,4,2,1978,3.453736,1,8,4.736292e+05
641,4688,5,3,1983,1.916716,2,7,1.004456e+06
941,4789,3,3,1998,4.118431,1,8,1.033817e+06


---

## Question 2 (10 Marks)

Handle missing values and perform feature engineering:

- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.


In [3]:
# Question 2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# 1. Drop columns with more than 50% missing values
threshold = 0.5 * len(df)
df_cleaned = df.dropna(thresh=threshold, axis=1)

# Separate features (X) and target (y)
X = df_cleaned.drop(columns=["House_Price"])
y = df_cleaned["House_Price"]

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# 2. Impute numerical columns using SimpleImputer with mean strategy
if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy="mean")
    X[num_cols] = num_imputer.fit_transform(X[num_cols])

# 3. Impute categorical columns using most_frequent strategy
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# 4. Perform Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

X_train shape: (800, 7)
X_test shape : (200, 7)
y_train shape: (800,)
y_test shape : (200,)


---

## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):

- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).


In [4]:
# Question 3
# Extracting Square_Footage as feature and House_Price as target
X_train_sf = X_train["Square_Footage"].values
y_train_sf = y_train.values
X_test_sf = X_test["Square_Footage"].values
y_test_sf = y_test.values

# Feature Scaling (Standardization) for numerical stability in Gradient Descent
mean_x = np.mean(X_train_sf)
std_x = np.std(X_train_sf)

X_train_scaled = (X_train_sf - mean_x) / std_x
X_test_scaled = (X_test_sf - mean_x) / std_x

# Initialize parameters
m_scaled = 0.0
c_scaled = 0.0
learning_rate = 0.01
epochs = 1000
n = len(X_train_scaled)

# Batch Gradient Descent
for iteration in range(epochs):
    y_pred_iter = m_scaled * X_train_scaled + c_scaled
    dm = (-2 / n) * np.sum(X_train_scaled * (y_train_sf - y_pred_iter))
    dc = (-2 / n) * np.sum(y_train_sf - y_pred_iter)
    m_scaled -= learning_rate * dm
    c_scaled -= learning_rate * dc

# Convert parameters back to original unscaled feature space (y = m*X + c)
m_original = m_scaled / std_x
c_original = c_scaled - (m_scaled * mean_x / std_x)

# Predictions on test set
y_test_pred_q3 = m_original * X_test_sf + c_original

print(f"Learned Slope (m): {m_original:.4f}")
print(f"Learned Intercept (c): {c_original:.4f}")
print("\nFirst 5 Predictions on Test Set:", np.round(y_test_pred_q3[:5], 2))
print("First 5 Actual Test Values    :", np.round(y_test_sf[:5], 2))

Learned Slope (m): 200.5569
Learned Intercept (c): 54229.0585

First 5 Predictions on Test Set: [ 858863.25  517515.44  998450.83 1043375.57  785459.43]
First 5 Actual Test Values    : [ 901000.49  494537.51  949404.2  1040389.05  794010.02]


---

## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:

- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`


In [5]:
# Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_features = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
cat_features = ["Neighborhood_Quality"]

column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ],
    remainder="drop",
)

print("ColumnTransformer constructed successfully:")
print(column_transformer)

ColumnTransformer constructed successfully:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Neighborhood_Quality'])])


---

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:

- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples


In [6]:
# Question 5
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Construct Pipeline
pipeline_sgd = Pipeline(
    steps=[
        ("preprocessor", column_transformer),
        ("regressor", SGDRegressor(random_state=42)),
    ]
)

# Train Pipeline
pipeline_sgd.fit(X_train, y_train)

# Predictions on Test Set
y_pred_sgd = pipeline_sgd.predict(X_test)

# Evaluation Metrics
rmse_sgd = np.sqrt(mean_squared_error(y_test, y_pred_sgd))
r2_sgd = r2_score(y_test, y_pred_sgd)

print(f"SGDRegressor RMSE    : {rmse_sgd:.4f}")
print(f"SGDRegressor R² Score: {r2_sgd:.4f}")

# Predicted vs Actual values for first 10 test samples
results_sgd = pd.DataFrame(
    {
        "Actual House Price": y_test.iloc[:10].values,
        "Predicted House Price": y_pred_sgd[:10],
    }
)
display(results_sgd)

SGDRegressor RMSE    : 29096.4609
SGDRegressor R² Score: 0.9869


,Actual House Price,Predicted House Price
0,9.010005e+05,8.513699e+05
1,4.945375e+05,5.073428e+05
2,9.494042e+05,9.858317e+05
3,1.040389e+06,1.022465e+06
4,7.940100e+05,7.598091e+05
5,7.240336e+05,7.656757e+05
6,9.984392e+05,1.004528e+06
7,9.097134e+05,9.035460e+05
8,7.926815e+05,7.971931e+05
9,9.474908e+05,8.886267e+05


---

## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:

- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples


In [7]:
# Question 6
from sklearn.linear_model import LinearRegression

# Construct Pipeline with LinearRegression
pipeline_lr = Pipeline(
    steps=[("preprocessor", column_transformer), ("regressor", LinearRegression())]
)

# Train Pipeline
pipeline_lr.fit(X_train, y_train)

# Predictions on Test Set
y_pred_lr = pipeline_lr.predict(X_test)

# Evaluation Metrics
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"LinearRegression RMSE    : {rmse_lr:.4f}")
print(f"LinearRegression R² Score: {r2_lr:.4f}")

# Predicted vs Actual values for first 10 test samples
results_lr = pd.DataFrame(
    {
        "Actual House Price": y_test.iloc[:10].values,
        "Predicted House Price": y_pred_lr[:10],
    }
)
display(results_lr)

LinearRegression RMSE    : 29093.0752
LinearRegression R² Score: 0.9869


,Actual House Price,Predicted House Price
0,9.010005e+05,8.514091e+05
1,4.945375e+05,5.076410e+05
2,9.494042e+05,9.859904e+05
3,1.040389e+06,1.022655e+06
4,7.940100e+05,7.606981e+05
5,7.240336e+05,7.657823e+05
6,9.984392e+05,1.005039e+06
7,9.097134e+05,9.034078e+05
8,7.926815e+05,7.979712e+05
9,9.474908e+05,8.888504e+05


---

## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)

### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:

- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)


In [8]:
# Question 7
# 1. Pick random 100 samples from the dataset
sample_100 = df.sample(n=100, random_state=42)

# 2. Extract features (Square_Footage, Num_Bedrooms, Num_Bathrooms) and target (House_Price)
X_sample = sample_100[["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]].values
y_sample = sample_100["House_Price"].values

# 3. Add intercept column (bias term x0 = 1) to X matrix
X_b = np.c_[np.ones((len(X_sample), 1)), X_sample]

# 4. Apply Normal Equation: θ = (XᵀX)⁻¹ Xᵀy
theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y_sample)

# 5. Print learned coefficients
print("Learned Coefficients (θ values):\n")
print(f"Intercept (θ0)        : {theta[0]:.4f}")
print(f"Square_Footage (θ1)   : {theta[1]:.4f}")
print(f"Num_Bedrooms (θ2)     : {theta[2]:.4f}")
print(f"Num_Bathrooms (θ3)    : {theta[3]:.4f}")

Learned Coefficients (θ values):

Intercept (θ0)        : 20888.2595
Square_Footage (θ1)   : 202.3082
Num_Bedrooms (θ2)     : 8303.2810
Num_Bathrooms (θ3)    : 3020.3285
